# Podcast RSS Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook pulls episode metadata from any podcast's RSS feed and exports it as structured data. Enter a podcast RSS URL and download a complete episode catalog with titles, publication dates, durations, season/episode numbers, descriptions, and audio URLs.

Every podcast published through major hosting platforms (Buzzsprout, Simplecast, Anchor, Libsyn, Podbean, etc.) has an RSS feed. This notebook reads that feed directly, requiring no API key or account.

## Key Features

1. **No API Key Required**: Reads standard podcast RSS feeds directly
2. **Complete Metadata**: Titles, dates, durations, season/episode numbers, descriptions, and audio URLs
3. **Show-Level Data**: Podcast title, author, description, artwork URL, and categories
4. **Multiple Feeds**: Process multiple podcast feeds in one session
5. **Structured Export**: Download episode catalogs as CSV or styled Excel

## Workflow

1. **Enter**: Paste one or more podcast RSS feed URLs
2. **Fetch**: Pull episode metadata from each feed
3. **Review**: Preview the episode catalog in a table
4. **Export**: Download structured data for further analysis

## Finding a Podcast's RSS Feed

Most podcast hosting platforms provide RSS feed URLs:
- **Apple Podcasts**: Search on [getrssfeed.com](https://getrssfeed.com/) or look in the podcast's page source
- **Spotify-hosted (Anchor)**: Usually `https://anchor.fm/s/[id]/podcast/rss`
- **Buzzsprout**: `https://feeds.buzzsprout.com/[id].rss`
- **Simplecast**: Check the podcast's website source for the RSS link
- **Libsyn**: `https://[showname].libsyn.com/rss`
- **Podbean**: `https://feed.podbean.com/[showname]/feed.xml`

You can also search for any podcast's RSS feed at [castos.com/tools/find-podcast-rss-feed](https://castos.com/tools/find-podcast-rss-feed/).

## Applications

This notebook supports research involving podcast media, including building episode catalogs for content analysis, tracking publication patterns over time, comparing podcast metadata across shows, and preparing structured data for media studies or digital ethnography projects.

## Methodological Positioning

This notebook represents a **computational anthropology** approach to media data collection. It retrieves publicly available podcast metadata but does not access or analyze audio content. Interpretation remains the work of the researcher.

## Target Audience

Designed for researchers working with podcast media, from graduate students conducting media analysis to research teams building podcast corpora for qualitative coding.

## Technical Approach

The notebook uses the **feedparser** library to parse standard podcast RSS feeds (RSS 2.0 with iTunes extensions). All processing runs locally in the notebook.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of notebooks supporting computational and AI-assisted approaches to anthropological research.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required packages and import libraries. Run this cell first.

In [ ]:
# Install required packages
!pip install feedparser pandas openpyxl ipywidgets -q

import re
import unicodedata
import feedparser
import pandas as pd
from datetime import datetime
from email.utils import parsedate_to_datetime
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def normalize_text(text):
    """Normalize unicode characters and strip HTML tags."""
    if not isinstance(text, str): return text
    # Strip HTML
    text = re.sub(r'<[^>]+>', '', text)
    text = unicodedata.normalize('NFKC', text)
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...'),('&amp;','&'),('&lt;','<'),('&gt;','>')]:
        text = text.replace(old, new)
    return text.strip()


def make_slug(query):
    """Create a clean filename slug."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30] or 'podcast'


def parse_duration(dur):
    """Normalize duration to HH:MM:SS format."""
    if not dur: return ''
    dur = str(dur).strip()
    # Already HH:MM:SS
    if re.match(r'^\d+:\d{2}:\d{2}$', dur): return dur
    # MM:SS
    if re.match(r'^\d+:\d{2}$', dur): return f'0:{dur}'
    # Seconds only
    try:
        secs = int(dur)
        h, m, s = secs // 3600, (secs % 3600) // 60, secs % 60
        return f'{h}:{m:02d}:{s:02d}'
    except ValueError:
        return dur


def extract_episodes(feed):
    """Extract episode data from a parsed feed."""
    rows = []
    show_title = feed.feed.get('title', 'Unknown')
    for e in feed.entries:
        enc = e.get('enclosures', [{}])
        audio = enc[0].get('href', '') if enc else ''

        # Parse date
        pub_date = e.get('published', '')
        date_iso = ''
        try:
            dt = parsedate_to_datetime(pub_date)
            date_iso = dt.strftime('%Y-%m-%d')
        except Exception:
            pass

        rows.append({
            'show': normalize_text(show_title),
            'title': normalize_text(e.get('title', '')),
            'date': date_iso or pub_date,
            'duration': parse_duration(e.get('itunes_duration', '')),
            'season': e.get('itunes_season', ''),
            'episode': e.get('itunes_episode', ''),
            'description': normalize_text(e.get('subtitle', '') or e.get('summary', ''))[:500],
            'link': e.get('link', ''),
            'audio_url': audio,
        })
    return rows


def extract_show_info(feed):
    """Extract show-level metadata from a feed."""
    f = feed.feed
    img = ''
    if isinstance(f.get('image'), dict):
        img = f['image'].get('href', '')
    return {
        'title': normalize_text(f.get('title', '')),
        'author': normalize_text(f.get('author', f.get('itunes_author', ''))),
        'description': normalize_text(f.get('subtitle', f.get('description', '')))[:500],
        'link': f.get('link', ''),
        'image_url': img,
        'language': f.get('language', ''),
        'categories': ', '.join(c.get('term', '') for c in f.get('tags', []) if c.get('term')),
        'total_episodes': str(len(feed.entries)),
    }


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)


print("\u2713 All packages loaded")
print(f"\u2713 Environment: {"Google Colab" if IN_COLAB else "Local Jupyter"}")
print("\U0001f399\ufe0f Ready to configure podcast feed retrieval!")

## Fetch Podcast Data

Enter one or more podcast RSS feed URLs. The notebook will retrieve all episode metadata and show-level information from each feed.

In [ ]:
# Podcast RSS Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f399\ufe0f Podcast RSS Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook pulls episode metadata from podcast RSS feeds and exports structured data.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Enter:</strong> Paste one or more RSS feed URLs below (one per line)</li>'
instructions_html += '<li><strong>Fetch:</strong> Click to retrieve episode metadata</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

feed_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f517 RSS Feed URLs</h3>')

feed_input = widgets.Textarea(
    value='',
    placeholder='Paste RSS feed URLs here, one per line\n\nExample:\nhttps://feeds.simplecast.com/BqbsxVfO',
    layout=widgets.Layout(width='100%', height='100px')
)

feed_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Enter one RSS feed URL per line. Most podcast hosting platforms provide RSS feeds. '
    'Search for your podcast\'s feed at <a href="https://castos.com/tools/find-podcast-rss-feed/" target="_blank">castos.com</a>.</p>'
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(description='Fetch Podcast Data', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))
progress = widgets.HTML(value='')
out = widgets.Output()


def on_fetch(_):
    out.clear_output()
    progress.value = ''

    urls = [u.strip() for u in feed_input.value.strip().split('\n') if u.strip()]
    if not urls:
        with out:
            print("\u26a0\ufe0f Please enter at least one RSS feed URL.")
        return

    with out:
        print(f"\U0001f399\ufe0f Fetching {len(urls)} podcast feed(s)...")
        print()

    all_episodes = []
    all_shows = []

    for i, url in enumerate(urls):
        progress.value = f'<span style="color: #274C77;">\u2713 Feed {i+1}/{len(urls)}...</span>'
        try:
            feed = feedparser.parse(url)
            if not feed.entries:
                with out:
                    print(f"  \u26a0\ufe0f No episodes found at: {url[:60]}...")
                continue

            show_info = extract_show_info(feed)
            all_shows.append(show_info)
            episodes = extract_episodes(feed)
            all_episodes.extend(episodes)

            with out:
                print(f'  \u2713 {show_info["title"]}: {len(episodes)} episodes')

        except Exception as e:
            with out:
                print(f"  \u274c Error fetching {url[:60]}: {type(e).__name__}: {e}")

    progress.value = ''

    if not all_episodes:
        with out:
            print("\n\u26a0\ufe0f No episodes retrieved. Check your feed URLs.")
        return

    ep_df = pd.DataFrame(all_episodes)
    show_df = pd.DataFrame(all_shows)

    with out:
        print(f"\n\u2713 Total: {len(ep_df)} episodes across {len(show_df)} show(s)")
        print()
        print("\U0001f4ca Episode Preview:")
        display(ep_df[['show', 'title', 'date', 'duration', 'season', 'episode']].head(15))

    # Export
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    slug = make_slug(show_df.iloc[0]['title']) if len(show_df) == 1 else 'podcasts'
    base = f'podcast_{slug}_{timestamp}'
    fmt = format_input.value

    if fmt == 'xlsx':
        filepath = f'{base}.xlsx'
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            ep_df.to_excel(writer, sheet_name='Episodes', index=False)
            show_df.to_excel(writer, sheet_name='Show Info', index=False)
        style_excel(filepath)
    else:
        filepath = f'{base}.csv'
        ep_df.to_csv(filepath, index=False)

    with out:
        print()
        print(f"\u2713 Saved: {filepath}")
        if IN_COLAB:
            colab_files.download(filepath)


fetch_btn.on_click(on_fetch)

display(widgets.VBox([
    instructions,
    feed_header,
    feed_input,
    feed_help,
    export_header,
    format_input,
    fetch_btn,
    progress,
    out,
]))